# Session 32: Project 4: A Prediction Model

Data Science with Python, Level 2, Stage 4: Applied ML

## Learning Objectives

By the end of this session you will be able to:

1. Frame a prediction problem as a business decision.
2. Build a full pipeline from raw data to prediction.
3. Choose a model with cross-validation.
4. Pick a threshold that matches the business goal.
5. Measure the final model once on the test set.
6. Explain the model in a short report.

## Where We Are

Stage 4 ends with your fourth portfolio project. Over the last eight sessions you built a churn model step by step: a preprocessor, model comparisons, imbalance handling, a pipeline, tuning and explanations. Today you put it together into one clean, honest project.

The deliverable is a notebook that a hiring manager or a retention team could read: what problem it solves, how good the model really is, how to use it, and what drives the predictions. You may use the churn data or another course dataset with your instructor's approval.

## Setup

Run this cell first. The 20 percent test set was split off in Session 24 with the same seed, and has not been used since.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.metrics import roc_auc_score, recall_score, precision_score, confusion_matrix
from sklearn.inspection import permutation_importance

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
churn = pd.read_csv(DATA_URL + "telecom_churn.csv")
y = (churn["churn"] == "Yes").astype(int)
X = churn.drop(columns=["customer_id", "churn"])
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2,
                                                stratify=y, random_state=0)

## 1. The Project Brief

| Dataset | Target | Decision it supports |
| --- | --- | --- |
| `telecom_churn.csv` | churn | Which customers should the retention team call? |
| `card_transactions.csv` | is_fraud | Which payments should be checked? |
| `apartments.csv` | price_k | What is a fair asking price for a flat? |
| `exam_scores.csv` | passed | Which students need extra support early? |

Regression projects replace the threshold milestone with an error analysis: where is the model most wrong?

## 2. The Modelling Workflow

| Step | What you do |
| --- | --- |
| 1. Frame | Target, decision, metric and the cost of each error |
| 2. Split | Set aside the test set first, stratified |
| 3. Pipeline | Preprocessing and model in one object |
| 4. Compare | Baseline and candidates with cross-validation |
| 5. Tune | Search the best candidate's settings |
| 6. Threshold | Choose it from out-of-fold predictions |
| 7. Test once | Fit on all development data, score the test set |
| 8. Explain | Importance, effects and a written recommendation |

In [ ]:
preprocess = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()),
     make_column_selector(dtype_include="number")),
    ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="None"),
                          OneHotEncoder(handle_unknown="ignore")),
     make_column_selector(dtype_include=["object", "string"])),
    ("bool", "passthrough", make_column_selector(dtype_include="bool")),
])
model = Pipeline([("prep", preprocess), ("model", LogisticRegression(C=1, max_iter=1000))])
scores = cross_val_score(model, X_dev, y_dev, cv=5,
                         scoring="roc_auc")
print(f"CV AUC {scores.mean():.3f} +/- {scores.std():.3f}")

## 3. Choosing the Threshold

In [ ]:
oof = cross_val_predict(model, X_dev, y_dev, cv=5,
                        method="predict_proba")[:, 1]

for t in [0.2, 0.3, 0.4, 0.5]:
    p = (oof >= t).astype(int)
    print(t, "recall", round(recall_score(y_dev, p), 2),
          "precision", round(precision_score(y_dev, p), 2))

In [ ]:
model.fit(X_dev, y_dev)
test_proba = model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= 0.3).astype(int)

print("Test AUC:", round(roc_auc_score(y_test, test_proba), 3))
print("Recall:", round(recall_score(y_test, test_pred), 2),
      " Precision:", round(precision_score(y_test, test_pred), 2))
print(confusion_matrix(y_test, test_pred))

In [ ]:
names = model["prep"].get_feature_names_out()
odds = pd.Series(np.exp(model["model"].coef_[0]), index=names).sort_values()
print(odds.round(2).iloc[[0, 1, 2, -3, -2, -1]])

| Item | Content |
| --- | --- |
| Purpose | Rank customers by risk of leaving in the next period |
| Data | 2,000 practice customers, 10 input columns, 30 percent churn |
| Model | Logistic regression in a scikit-learn pipeline |
| Performance | Test AUC 0.80; at threshold 0.3: recall 0.78, precision 0.53 |
| Main drivers | Monthly contract, short tenure, high charges, support calls |
| Limits | Practice data; associations, not causes; retrain as offers change |

## 4. How the Project Is Graded

| Criterion | Points | What earns full points |
| --- | --- | --- |
| Framing | 10 | Clear target, decision, metric and error costs |
| Pipeline | 20 | All preprocessing inside one pipeline, no leakage |
| Model choice | 20 | Baseline, at least two candidates, cross-validation, tuning |
| Evaluation | 20 | Threshold from validation, test set used once, right metrics |
| Explanation | 20 | Importance or coefficients, effects, plain-language findings |
| Reproducible | 10 | Runs top to bottom; random seeds set |

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Peeking | Checking the test score after every change | Use the test set once, at the very end. |
| No baseline | Reporting AUC 0.80 without context | Compare with a majority-class or simple model. |
| Threshold 0.5 by habit | Ignoring what the business needs | Choose the threshold from the goal and costs. |
| Jargon | Writing "AUC improved 0.004" for managers | Translate: "we find 94 of 121 leavers". |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Frame the Problem

Write the framing for your project in a text cell: target, positive class, decision, main metric, and the cost of a false positive and a false negative. Print it from a code cell.

In [ ]:
# Write your code below

### In-Class Exercise 2: Baseline First

Compute two baselines on the development set: the majority-class accuracy, and the cross-validated AUC of a logistic regression using only `tenure_months`.

In [ ]:
# Write your code below

### In-Class Exercise 3: Call List

Using the final model, list the 10 test customers with the highest predicted risk, with their contract, tenure, charges and whether they really left. How many of the 10 did leave?

In [ ]:
# Write your code below

### In-Class Exercise 4: Peer Review

Swap notebooks with a classmate. Using the rubric, write two strengths and two improvements as comments, and check one thing carefully: is the test set used only once?

In [ ]:
# Write your code below

## Summary

- Frame the problem as a decision with costs.
- Split off the test set first and keep it untouched.
- Put every step in one pipeline and compare with a baseline.
- Choose the threshold from out-of-fold predictions.
- Score the final model once, then explain it in plain words.
- A model card states purpose, performance and limits.

## What's Next

Level 2 continues with Stage 5, deployment. Your projects deserve to be seen. Next session you will learn Git, the tool every developer uses to save versions of their work, and publish your first repository on GitHub.